# Fase de entendimiento de los datos: EDA

Análisis exploratorio del dataset generado a partir de fotografías HEIC de surtidores Shell. Se evalúan estructura, calidad, distribución, evolución temporal, consistencia de precios y metadata disponible.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

CSV_PATH = ROOT / 'output' / 'gas_prices.csv'
if not CSV_PATH.exists():
    subprocess.run([sys.executable, str(ROOT / 'src' / 'gas_pipeline.py')], check=True)

df = pd.read_csv(CSV_PATH)
df['fecha'] = pd.to_datetime(df['fecha_original'], format='%Y:%m:%d %H:%M:%S')
df = df.sort_values('fecha').reset_index(drop=True)
print(f'Observaciones: {len(df)} | Variables: {df.shape[1]}')
print(f'Periodo: {df.fecha.min():%d/%m/%Y} - {df.fecha.max():%d/%m/%Y}')

## 1. Estructura y significado de los datos

Cada fila representa una fotografía. Las lecturas manuales son la referencia visual; los campos estimados provienen de las reglas del negocio. Los flags valen `1` cuando la diferencia está dentro de la tolerancia y `0` cuando requiere revisión.

In [ ]:
columnas_clave = [
    'archivo', 'fecha', 'gps_lat', 'gps_lon',
    'precio_diesel_imagen_manual', 'precio_regular_imagen_manual',
    'precio_super_imagen_manual', 'precio_vpower_imagen_manual',
    'precio_super_division', 'todos_match_flag', 'requiere_revision_flag'
]
df[columnas_clave]

In [ ]:
campos_control = [
    'fecha_original', 'gps_lat', 'gps_lon', 'total_venta_manual', 'galones_manual',
    'precio_diesel_imagen_manual', 'precio_regular_imagen_manual',
    'precio_super_imagen_manual', 'precio_vpower_imagen_manual'
]
calidad = pd.DataFrame({
    'tipo': df[campos_control].dtypes.astype(str),
    'nulos': df[campos_control].isna().sum(),
    'unicos': df[campos_control].nunique(),
})
print(f'Archivos duplicados: {df.archivo.duplicated().sum()}')
calidad

In [ ]:
price_columns = {
    'precio_diesel_imagen_manual': 'Diesel',
    'precio_regular_imagen_manual': 'Regular',
    'precio_super_imagen_manual': 'Súper',
    'precio_vpower_imagen_manual': 'V-Power',
}
df[list(price_columns)].describe().round(2).rename(columns=price_columns)

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#202020', '#d7a900', '#14804a', '#d83232']
for (column, label), color in zip(price_columns.items(), colors):
    ax.plot(df['fecha'], df[column], marker='o', linewidth=2.2, label=label, color=color)
ax.set(title='Evolución de precios por galón', xlabel='Fecha', ylabel='Quetzales (Q)')
ax.legend(ncol=4, frameon=False)
fig.autofmt_xdate()
plt.show()

In [ ]:
reglas = pd.DataFrame({
    'fecha': df['fecha'].dt.date,
    'regular_menos_super': df['precio_regular_imagen_manual'] - df['precio_super_imagen_manual'],
    'vpower_menos_super': df['precio_vpower_imagen_manual'] - df['precio_super_imagen_manual'],
    'dif_regular': df['diferencia_regular'],
    'dif_super': df['diferencia_super'],
    'dif_vpower': df['diferencia_vpower'],
    'flag_global': df['todos_match_flag'],
})
reglas

In [ ]:
metadata_rows = []
for path in sorted((ROOT / 'output' / 'metadata').glob('*.json')):
    item = json.loads(path.read_text(encoding='utf-8'))
    metadata_rows.append({
        'archivo': item.get('FileName'),
        'camara': item.get('Model'),
        'campos_metadata': len(item),
        'ancho': item.get('ExifImageWidth'),
        'alto': item.get('ExifImageHeight'),
        'iso': item.get('ISO'),
    })
metadata_df = pd.DataFrame(metadata_rows)
metadata_df

In [ ]:
inicio, fin = df.iloc[0], df.iloc[-1]
resumen = pd.DataFrame({
    'combustible': list(price_columns.values()),
    'precio_inicial': [inicio[c] for c in price_columns],
    'precio_final': [fin[c] for c in price_columns],
})
resumen['cambio'] = (resumen['precio_final'] - resumen['precio_inicial']).round(2)
print(f"Flags correctos: {int(df['todos_match_flag'].sum())}/{len(df)}")
print(f"Revisiones pendientes: {int(df['requiere_revision_flag'].sum())}")
resumen

## 2. Hallazgos y límites

- Las cinco observaciones tienen metadata, GPS, cuatro pantallas detectadas y ninguna revisión pendiente.
- Regular, Súper y V-Power cumplen las reglas definidas; la mayor diferencia por redondeo es Q0.01.
- En el periodo, Regular, Súper y V-Power aumentaron Q4.00; Diesel aumentó Q4.40.
- Las coordenadas y la cámara son consistentes, por lo que la muestra representa una misma ubicación y dispositivo.
- **Limitación:** cinco fotografías de una sola estación permiten validar el pipeline y describir cambios, pero no permiten inferencia estadística ni pronósticos.